# Demo 03 — Metrics That Matter: Can We Trust the Citations?

**Audience:** customer executives, product managers, compliance / audit officers.

**What this notebook shows:** a plain-English read-out of **three metrics** and **two caveats** that together answer the grown-up version of *"does it work?"* — without requiring you to read the technical evaluation notebook.

**Prerequisites:** you've already seen
- `demo_01_the_problem.ipynb` — *why* sentence-level citation matters.
- `demo_02_*.ipynb` — *what* a cited answer looks like end-to-end.

> ✅ **No Azure calls. No LLM calls. No secrets required.** This notebook reads only cached evaluation results from `data/notebook_cache/eval/`. It will run on any laptop, offline.


## The three metrics — in plain English

If you only remember three things about whether a citation system is trustworthy, remember these:

| Metric | The question it answers | Whose question is this? |
| --- | --- | --- |
| **Faithfulness** | When the system cites a sentence, does that sentence actually *say* what the answer claims? | The **auditor's** question. |
| **Stability** (self-consistency) | Ask the same question twice — do you get the same answer? | The **trust** question. |
| **Coverage** | What fraction of the answer's sentences have **any** citation attached? | The **completeness** question. |

> 🎯 **These three together are what an auditor needs to sign off.** Faithfulness proves you didn't hallucinate. Stability proves the system is reliable. Coverage proves you didn't leave claims un-sourced.

A fourth number — **F1** — shows up later. Read the caveat in Part 3 before you draw conclusions from it.


## Part 1 — Load the cached evaluation

We read a small JSON manifest produced by the Phase 1a smoke evaluation run. Five questions, two citation strategies, no live calls.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

CACHE = Path("../data/notebook_cache/eval")
manifest = json.loads((CACHE / "manifest.json").read_text())

run_id = manifest.get("run_id", "(unknown)")
n_items = next(iter(manifest["strategies"].values())).get("n_items", "?")
elapsed = manifest.get("elapsed_seconds")

display(Markdown(
    f"**Run:** `{run_id}`  \n"
    f"**Questions:** {n_items}  \n"
    f"**Strategies compared:** {', '.join(manifest['strategies'].keys())}  \n"
    f"**Elapsed (original run):** {elapsed:.1f}s" if elapsed else ""
))

In [ ]:
def _row(strategy_name, s):
    return {
        "Strategy": strategy_name,
        "Faithfulness (%)": round(s.get("macro_percent_faithful", float("nan")), 1),
        "Stability": round(s.get("macro_stability", float("nan")), 3),
        "Coverage": round(s.get("macro_coverage", float("nan")), 3),
        "F1 (exact-sid, see caveat)": round(s.get("macro_f1", float("nan")), 3),
        "Retrieval R@k": round(s.get("macro_retrieval_recall_at_k", float("nan")), 3),
    }

strategy_rows = [_row(name, s) for name, s in manifest["strategies"].items()]
summary_df = pd.DataFrame(strategy_rows).set_index("Strategy")
display(Markdown("### Phase 1a smoke results (5 questions × 2 strategies)"))
summary_df

## Part 2 — Strategy A vs Strategy B

Two candidate citation strategies were compared on the same five questions:

- **Strategy A — `inline_prompted`.** The model is *asked* to produce sentence-level citations inline as it writes the answer.
- **Strategy B — `post_gen_alignment`.** The model writes an answer first; a second pass then attaches citations by aligning answer sentences to retrieved evidence.

### What we see, metric by metric

- **Faithfulness.** Strategy A is ~84% faithful; Strategy B ~44%. When Strategy B does attach a citation, that citation too often does *not* actually support the claim. For an auditor this is the single most important number on the page.
- **Stability.** Strategy A (~0.72) is markedly more self-consistent than Strategy B (~0.38). Strategy B's post-hoc alignment is noisy — the same question can produce different citations on re-run.
- **Coverage.** Strategy A cites **every** answer sentence (1.00). Strategy B cites fewer than half (0.44), leaving claims un-sourced — which for a compliance reviewer is a non-starter.

### Operational takeaway

> **Strategy A wins on every auditor-facing metric in this smoke run.** Strategy B's `cite_align` post-processor is not yet robust enough to ship. The eval harness *catches this before a customer sees it* — which is exactly its job.


In [ ]:
metrics_to_plot = ["Faithfulness (%)", "Stability", "Coverage"]
plot_df = summary_df[metrics_to_plot].copy()
plot_df["Faithfulness (%)"] = plot_df["Faithfulness (%)"] / 100.0  # normalise to 0-1 for visual comparability

try:
    import matplotlib.pyplot as plt  # noqa: F401
    ax = plot_df.T.plot(kind="bar", figsize=(9, 4.5), rot=0,
                        color=["#2b7a4b", "#b05a1c"], edgecolor="black")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Score (0–1 scale; faithfulness rescaled from %)")
    ax.set_title("Strategy A vs Strategy B — auditor-facing metrics")
    ax.axhline(1.0, color="gray", linestyle=":", linewidth=0.8)
    ax.legend(title="Strategy", loc="lower right")
    for container in ax.containers:
        ax.bar_label(container, fmt="%.2f", padding=2, fontsize=8)
    plt.tight_layout()
    plt.show()
except Exception:
    display(plot_df.style.format("{:.2f}").background_gradient(axis=None, cmap="Greens"))

## Part 3 — ⚠️ The F1-pessimism caveat (read this before quoting F1)

> **F1 on this page is a *floor*, not a true score.**

The F1 column shows **0.27 for Strategy A** and **0.00 for Strategy B**. At face value that looks alarming. It isn't — here's why:

- F1 here is computed against a synthetic ground-truth set of citations, using **exact sentence-ID (sid) matching**.
- Legal and tax documents routinely contain **parallel sentences** — two sentences that say substantively the same thing but live at different sids (e.g., in a regulation and its implementing guidance, or across versions).
- A human auditor reviewing the answer would **accept** either parallel sentence as a valid citation. Our harness will only accept the *exact* sid the synthetic GT happened to pick.
- So every parallel-sentence case scores as a miss, even though the citation is correct in substance.

**Operational reading:**
- `faithfulness` and `stability` — trust these. They measure what they say.
- `F1 (exact-sid)` — treat as a **lower bound**. A human-graded F1 with "parallel sid acceptable" semantics would be substantially higher. Phase 1b adds that grading.

In short: when an F1 of 0.00 is paired with a faithfulness of 44% (Strategy B), the low F1 is *partially* a metric artefact, but the faithfulness gap is real. When F1 of 0.27 is paired with faithfulness of 84% (Strategy A), the metric artefact dominates — Strategy A is doing much better than 0.27 suggests.


## Part 4 — ⚠️ The non-SME caveat (read this before showing a customer)

> **None of the results above have been validated by a domain expert.**

Everything on this page was produced and reviewed by ML engineers and PMs — not by tax auditors, not by compliance officers, not by any domain subject-matter expert (SME). That means:

- The synthetic ground-truth answers were generated and spot-checked by laypeople.
- The "faithfulness" judge is an LLM, not an auditor.
- The corpus is a small Phase 1a sample, not a production document set.

**Phase 1b brings in tax-domain SMEs** to produce human-graded ground truth, human faithfulness labels, and human adjudication of parallel-sentence cases. Until that lands:

- ✅ Use these numbers to compare **strategies against each other** (that's what the harness is *for*).
- ✅ Use these numbers to decide what to ship to **SME review next**.
- ❌ Do **not** present these numbers to a customer as **domain-validated** results.

See `docs/status.md` and the *layperson review* section of `docs/phase-1a/evaluation.md` for the full caveat trail.


## Part 5 — What this unlocks

With an evaluation harness that produces these three metrics on every run, we can:

- **Sign-off workflow.** Regression-test every model, prompt, and retrieval-config change against a held-out question set before shipping.
- **Confidence-building.** Give auditors and customers a repeatable, defensible scorecard — not a demo reel.
- **Phase 2 agentic features.** Multi-hop reasoning, tool use, and cross-document synthesis only make sense once single-hop citation is trustworthy. These metrics are the gate.

### We still need

- 🧑‍⚖️ **SME ground truth** — human-authored expected citations, not synthetic.
- 🧑‍⚖️ **SME faithfulness labels** — domain experts, not LLM judges.
- 📚 **Broader corpora** — beyond the Phase 1a sample, onto realistic customer document sets.
- 📈 **Parallel-sentence-aware F1** — so F1 stops being a pessimistic floor.


## Where to go next

- 📖 **Full docs site** — the published GitHub Pages site under `docs/` for the end-to-end story.
- 🧪 **Technical flavour** — `notebooks/05_evaluation.ipynb` is the engineering view of the same run (all the plumbing, all the field names, all the knobs).
- 🗺️ **Roadmap** — `docs/status.md` tracks what's in Phase 1a, what's coming in Phase 1b (SME validation), and what Phase 2 agentic work depends on these metrics.

---
*No Azure calls were made in the rendering of this notebook.*
